# PopDAM Bulk Vision Tagging Server (Google Colab)

Runs an OpenAI-compatible, continuously batched **vLLM** server for PopDAM bulk image tagging (`Qwen/Qwen3-VL-32B-Instruct`) on a high-VRAM Colab GPU (G4 96 GB / A100 80 GB / H100 80 GB) behind a temporary authenticated HTTPS tunnel.

- **Zero database credentials in Colab:** PopDAM's Railway worker keeps all Supabase access, tagging rules, character binding, and cursor resume.
- **Automatic idle shutdown:** After `IDLE_SHUTDOWN_MINUTES` with no incoming tagging requests, this notebook automatically disconnects the Colab runtime so compute units are never wasted.

In [ ]:
# === 1. Configuration ===
MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = 0.90
IDLE_SHUTDOWN_MINUTES = 20  # Auto-disconnect Colab runtime after this many idle minutes


In [ ]:
# === 2. Install vLLM + cloudflared, launch server + tunnel, and run idle watchdog ===
import json, os, re, secrets, subprocess, time, urllib.request

print("=== GPU ===")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"]).decode().strip())

print("\n=== Installing vLLM and cloudflared ===")
subprocess.check_call(["pip", "install", "-q", "-U", "vllm", "qwen-vl-utils"])
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.check_call([
        "curl", "-fsSL",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", "/usr/local/bin/cloudflared",
    ])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

API_KEY = "colab_" + secrets.token_urlsafe(32)
vllm_log = open("/tmp/vllm.log", "w")
vllm_proc = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api-key", API_KEY,
    "--dtype", "bfloat16",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--limit-mm-per-prompt", "image=1",
], stdout=vllm_log, stderr=subprocess.STDOUT)

print(f"\n=== Starting vLLM for {MODEL_ID} (takes ~2-4 min on first download) ===")
ready = False
for _ in range(360):
    if vllm_proc.poll() is not None:
        raise RuntimeError("vLLM exited early; check /tmp/vllm.log")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)
if not ready:
    raise RuntimeError("Timed out waiting for vLLM /health")

cf_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
tunnel_url = None
for line in cf_proc.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break
if not tunnel_url:
    raise RuntimeError("Failed to obtain Cloudflare tunnel URL")

print("\n=== READY FOR POPDAM ===")
print("Set admin_config.COLAB_VISION_CONFIG to:")
print(json.dumps({"base_url": tunnel_url, "api_key": API_KEY}, indent=2))
print(f"\nPopDAM model ID: colab-direct/{MODEL_ID}")
print(f"Idle auto-shutdown armed: {IDLE_SHUTDOWN_MINUTES} minutes without requests.")

def read_request_count():
    try:
        txt = urllib.request.urlopen("http://127.0.0.1:8000/metrics", timeout=3).read().decode()
        total = 0.0
        for line in txt.splitlines():
            if line.startswith("vllm:request_success_total") or line.startswith("vllm:num_requests_running"):
                parts = line.rsplit(" ", 1)
                if len(parts) == 2:
                    total += float(parts[1])
        return total
    except Exception:
        return -1.0

last_count = read_request_count()
last_activity = time.time()
try:
    while True:
        time.sleep(30)
        if vllm_proc.poll() is not None or cf_proc.poll() is not None:
            print("Server or tunnel stopped; shutting down.")
            break
        current = read_request_count()
        if current != last_count and current >= 0:
            last_count = current
            last_activity = time.time()
        idle_min = (time.time() - last_activity) / 60.0
        if idle_min >= IDLE_SHUTDOWN_MINUTES:
            print(f"Idle for {idle_min:.1f} min >= {IDLE_SHUTDOWN_MINUTES} min. Unassigning Colab runtime...")
            break
finally:
    cf_proc.terminate()
    vllm_proc.terminate()
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass
